# 008 QA With Evidence And Validation

这一课把上一课得到的 `evidence_package.json` 变成一个可追溯回答。

你要掌握三件事：

1. 为什么不能把召回结果原样塞给模型。
2. 如何把文本证据和图证据压缩成 prompt 上下文。
3. 如何对模型回答做最低限度验证：是否引用证据、引用是否存在、是否承认信息不足。


## 1. 本课的位置

前面 001-007 完成的是“知识库准备”和“证据召回”。

这一课开始进入真正问答：

```text
用户问题
  -> 多路召回
  -> evidence_package
  -> 证据压缩
  -> 问答模型
  -> 回答验证
```

如果用你熟悉的 Java 业务代码类比：

- 召回层像 DAO：负责把相关记录查出来。
- 证据包像 QueryResult DTO：统一承载不同来源的数据。
- 问答层像 Service：负责组织业务含义。
- 验证层像 Validator：检查响应有没有越权、缺字段、乱引用。


## 2. 导入依赖

本课只需要读取上一课生成的 JSON，并调用一次兼容 OpenAI 协议的本地模型网关。


In [1]:
import importlib.metadata
import json
import os
import re
from hashlib import sha1
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from openai import OpenAI

print('openai', importlib.metadata.version('openai'))

openai 2.36.0


## 3. 读取项目配置和证据包

这里继续使用样例 PDF 的 hash 作为 `doc_id`，这样能稳定定位到前面课时生成的目录。


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

SAMPLE_PDF = PROJECT_ROOT / 'raw' / '北京市密云水库防御洪水方案.pdf'
if not SAMPLE_PDF.exists():
    raise FileNotFoundError(SAMPLE_PDF)

doc_id = sha1(SAMPLE_PDF.read_bytes()).hexdigest()[:16]
generated_dir = PROJECT_ROOT / 'notebooks' / 'rag' / 'generated' / doc_id
evidence_package_path = generated_dir / 'evidence_package.json'
qa_result_path = generated_dir / 'qa_result.json'

if not evidence_package_path.exists():
    raise FileNotFoundError(f'请先执行第 007 课生成 evidence_package.json: {evidence_package_path}')

evidence_package = json.loads(evidence_package_path.read_text(encoding='utf-8'))

print('doc_id:', doc_id)
print('question:', evidence_package['question'])
print('text_evidence_count:', len(evidence_package.get('text_evidence', [])))
print('graph_evidence_count:', len(evidence_package.get('graph_evidence', [])))

doc_id: 63b7d4d0675426b5
question: 密引水渠引水流量不能满足泄洪要求应该怎么处理？
text_evidence_count: 6
graph_evidence_count: 10


## 4. 查看证据包结构

问答模型不应该直接面对完整数据库记录。

原因有三个：

1. 太长，浪费上下文。
2. 字段太杂，模型不知道哪些字段重要。
3. 没有证据编号，后续无法验证引用。

所以我们先把证据包转换成紧凑文本。


In [3]:
pprint({
    'question': evidence_package['question'],
    'text_evidence_ids': [item['chunk_id'] for item in evidence_package.get('text_evidence', [])],
    'graph_evidence_count': len(evidence_package.get('graph_evidence', [])),
})

{'graph_evidence_count': 10,
 'question': '密引水渠引水流量不能满足泄洪要求应该怎么处理？',
 'text_evidence_ids': ['63b7d4d0675426b5_chunk_0029',
                       '63b7d4d0675426b5_chunk_0017',
                       '63b7d4d0675426b5_chunk_0102',
                       '63b7d4d0675426b5_chunk_0021',
                       '63b7d4d0675426b5_chunk_0023',
                       '63b7d4d0675426b5_chunk_0041']}


## 5. 压缩文本证据和图证据

这里给每条证据分配稳定编号：

- `[T1]`、`[T2]`：文本 chunk 证据。
- `[G1]`、`[G2]`：图谱三元组证据。

模型回答时必须引用这些编号。


In [4]:
def normalize_space(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()


def clip_text(text: str, max_chars: int = 520) -> str:
    text = normalize_space(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + '...'


def build_evidence_context(package: dict, text_limit: int = 6, graph_limit: int = 6) -> tuple[str, dict]:
    lines = []
    citation_map = {}

    lines.append('【文本证据】')
    for index, item in enumerate(package.get('text_evidence', [])[:text_limit], start=1):
        citation_id = f'T{index}'
        citation_map[citation_id] = {
            'type': 'text',
            'chunk_id': item.get('chunk_id'),
            'page_start': item.get('page_start'),
            'page_end': item.get('page_end'),
        }
        lines.append(
            f'[{citation_id}] 页码 {item.get("page_start")}-{item.get("page_end")} '
            f'chunk={item.get("chunk_id")}：{clip_text(item.get("text", ""))}'
        )

    lines.append('')
    lines.append('【图谱证据】')
    for index, item in enumerate(package.get('graph_evidence', [])[:graph_limit], start=1):
        citation_id = f'G{index}'
        citation_map[citation_id] = {
            'type': 'graph',
            'subject': item.get('subject'),
            'predicate': item.get('predicate'),
            'object': item.get('object'),
            'chunk_id': item.get('chunk_id'),
            'page_start': item.get('page_start'),
            'page_end': item.get('page_end'),
        }
        lines.append(
            f'[{citation_id}] 页码 {item.get("page_start")}-{item.get("page_end")} '
            f'{item.get("subject")} --{item.get("predicate")}--> {item.get("object")}；'
            f'原文片段：{clip_text(item.get("evidence", ""), max_chars=220)}'
        )

    return '\n'.join(lines), citation_map


evidence_context, citation_map = build_evidence_context(evidence_package)
print(evidence_context[:2400])
print('\nknown citations:', sorted(citation_map))

【文本证据】
[T1] 页码 24-24 chunk=63b7d4d0675426b5_chunk_0029：图7 预报调度2（最大下泄流量3000 m3/s）过程示例 表 7 不同调度措施的调洪成果 预泄流 入库洪 最高水 最大出库流 最高库 超汛限 第10 调度模 量 峰 位出现 削峰率 量 水位 水位时 日水位 式 （m³/s （m³/s 时刻 （%） （m³/s） （m） 间（d） (m) ） ） （h） 规程调 200 17300 15400 157.46 10 62 153.74 11.0 度 预报调 200 17300 4000 157.50 9 88 155.38 76.9 度 200 17300 3000 157.72 9 88 155.74 82.7 4.3.4 超标准洪水调度 当水库发生超标准洪水时，敞开各输、泄水建筑物泄洪。 若仍不能满足工程安全的要求，经北京市水务局批准后，选 择炸开副坝的形式进行泄洪。调度规程详见《2024 年北京市 密云水库超标洪水防御预案》。 4.3.5 供水设施调度规程 密云水库白河发电隧洞以下有七孔桥节制闸和调节池， 其中七孔桥节制闸由北京市南水北调团城湖管理处负责日 常调度，调节池由北京市密云水库管理处负责调度。 （1）南水北调七孔桥节制闸调度规程 20
[T2] 页码 12-12 chunk=63b7d4d0675426b5_chunk_0017：2.2.2 调节池控制水位 6 月1 日至9 月30 日水位不超过90.50m。 3 调度运用计划 6 月1 日至8 月10 日水库限制水位为152.00m，相应库 容30.370 亿m3。8 月11 日至9 月30 日水库限制水位为 154.00m，相应库容33.610 亿m3。其中，8 月11 日至8 月 20 日为过渡期，将库水位控制在152.00m 至154.00m，在过 渡期内逐步抬高。 3.1 6 月1 日至8 月10 日 3.1.1 正常调度规程 （1）当水库水位未达到汛限水位152.00m 时，不泄洪。 （2）当水库水位已达到或超过汛限水位152.00m 但低 于154.32m 时，水库控泄流量600m3/s（包含京密引水渠引 水流量50m3/s）。 （3）当水库水位达到154.32m 但不超过157.50m 时， 水库控泄流量1550m3/s（包含京

## 6. 构造问答 Prompt

这一课的关键不是“让模型自由发挥”，而是给模型明确边界：

1. 只能使用证据回答。
2. 每个关键结论都要带引用。
3. 证据不足时必须说不足。
4. 不要把图谱关系当作比原文更高等级的事实，图谱只是结构化线索。


In [5]:
def build_qa_messages(question: str, evidence_context: str) -> list[dict]:
    system_prompt = (
        '你是一个严谨的 RAG 问答助手。'
        '只能根据用户提供的证据回答。'
        '回答中的关键结论必须引用证据编号，例如 [T1] 或 [G2]。'
        '如果证据不足以回答，请明确说“证据不足”。'
        '不要编造证据中没有的信息。'
    )
    user_prompt = f'''问题：{question}

证据：
{evidence_context}

请用中文回答，要求：
1. 先给出 3-6 条要点。
2. 每条要点后面带证据编号。
3. 最后补一句“证据覆盖范围”，说明哪些内容来自文本证据，哪些内容来自图谱证据。'''
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]

messages = build_qa_messages(evidence_package['question'], evidence_context)
print(messages[0]['content'])
print(messages[1]['content'][:1200])

你是一个严谨的 RAG 问答助手。只能根据用户提供的证据回答。回答中的关键结论必须引用证据编号，例如 [T1] 或 [G2]。如果证据不足以回答，请明确说“证据不足”。不要编造证据中没有的信息。
问题：密引水渠引水流量不能满足泄洪要求应该怎么处理？

证据：
【文本证据】
[T1] 页码 24-24 chunk=63b7d4d0675426b5_chunk_0029：图7 预报调度2（最大下泄流量3000 m3/s）过程示例 表 7 不同调度措施的调洪成果 预泄流 入库洪 最高水 最大出库流 最高库 超汛限 第10 调度模 量 峰 位出现 削峰率 量 水位 水位时 日水位 式 （m³/s （m³/s 时刻 （%） （m³/s） （m） 间（d） (m) ） ） （h） 规程调 200 17300 15400 157.46 10 62 153.74 11.0 度 预报调 200 17300 4000 157.50 9 88 155.38 76.9 度 200 17300 3000 157.72 9 88 155.74 82.7 4.3.4 超标准洪水调度 当水库发生超标准洪水时，敞开各输、泄水建筑物泄洪。 若仍不能满足工程安全的要求，经北京市水务局批准后，选 择炸开副坝的形式进行泄洪。调度规程详见《2024 年北京市 密云水库超标洪水防御预案》。 4.3.5 供水设施调度规程 密云水库白河发电隧洞以下有七孔桥节制闸和调节池， 其中七孔桥节制闸由北京市南水北调团城湖管理处负责日 常调度，调节池由北京市密云水库管理处负责调度。 （1）南水北调七孔桥节制闸调度规程 20
[T2] 页码 12-12 chunk=63b7d4d0675426b5_chunk_0017：2.2.2 调节池控制水位 6 月1 日至9 月30 日水位不超过90.50m。 3 调度运用计划 6 月1 日至8 月10 日水库限制水位为152.00m，相应库 容30.370 亿m3。8 月11 日至9 月30 日水库限制水位为 154.00m，相应库容33.610 亿m3。其中，8 月11 日至8 月 20 日为过渡期，将库水位控制在152.00m 至154.00m，在过 渡期内逐步抬高。 3.1 6 月1 日至8 月10 日 3.1.1 正常调度规程 （1）当水库水位未达到汛限水位152.00m 时，不

## 7. 调用问答模型

这里使用你当前网关里的 `qwen2.5-0.5b-instruct`。

小模型的优点是便宜、快、容易本地部署；缺点是遵循复杂格式的能力较弱。所以后面必须做验证。


In [7]:
RAG_CONFIG = {
    'model_base_url': os.getenv('RAG_MODEL_BASE_URL', 'http://192.168.102.19:8082/v1'),
    #'qa_model': os.getenv('RAG_QA_MODEL', 'qwen2.5-0.5b-instruct'),
    'qa_model': 'qwq',
}

RUN_QA_MODEL = True


def fallback_answer(package: dict) -> str:
    text_items = package.get('text_evidence', [])
    graph_items = package.get('graph_evidence', [])
    parts = []
    if text_items:
        parts.append('密云水库防御洪水方案包含编制目的、编制依据、适用范围、调度权限与职责、应急响应、附表附图等内容。[T1][T3]')
        parts.append('方案还涉及洪水调度请示、调度通知单、防汛调度令、主要技术指标、雨水情监测、洪水成果等附件材料。[T1]')
        parts.append('从调度职责看，密云水库管理处负责日常运行维护、雨水工情监测、洪水预报、调度方案制定和组织实施。[T4]')
    if graph_items:
        parts.append('图谱证据显示，该文档目录关联到防洪抢险预案和洪水调度方案等内容项。[G1][G2]')
    parts.append('证据覆盖范围：文本证据主要覆盖原文段落和页码，图谱证据主要提供目录和关系线索。')
    return '\n'.join(f'- {part}' for part in parts)


if RUN_QA_MODEL:
    client = OpenAI(api_key=os.getenv('RAG_MODEL_API_KEY', 'EMPTY'), base_url=RAG_CONFIG['model_base_url'])
    try:
        response = client.chat.completions.create(
            model=RAG_CONFIG['qa_model'],
            messages=messages,
            temperature=0.1,
            max_tokens=900,
        )
        answer = response.choices[0].message.content or ''
        answer_source = 'model'
    except Exception as exc:
        print('model call failed, using fallback answer:', repr(exc))
        answer = fallback_answer(evidence_package)
        answer_source = 'fallback'
else:
    answer = fallback_answer(evidence_package)
    answer_source = 'fallback'

print('answer_source:', answer_source)
print(answer)

answer_source: model
根据提供的证据，关于“密引水渠引水流量不能满足泄洪要求”的处理方式，证据中并未直接给出针对该特定情况的独立处理措施，但提供了相关的调度原则和超标准洪水下的应急方案。以下是基于证据的要点：

1. **常规控泄流量包含引水流量**：在正常调度规程中，水库的控泄流量是包含京密引水渠引水流量的。例如，当水库水位在152.00m至154.32m之间时，控泄流量为600m³/s，其中已包含京密引水渠引水流量50m³/s [T2]。
2. **高水位时全开闸门不控泄**：当水库水位超过157.50m时，输、泄水建筑物闸门全部开启，不再进行控泄，此时引水流量对泄洪能力的限制不再适用 [T2]。
3. **超标准洪水下的极端措施**：若发生超标准洪水，且敞开各输、泄水建筑物泄洪仍不能满足工程安全要求时，经北京市水务局批准后，可选择炸开副坝的形式进行泄洪 [T1]。
4. **利用预报调度预泄**：通过实施预报调度，根据洪水预报结果提前预泄，以降低库水位，从而为后续泄洪腾出库容。例如在预报调度模式下，最大下泄流量可达3000m³/s或4000m³/s，远高于常规调度 [T1]。
5. **证据中未明确具体替代方案**：虽然图谱证据[G1]提到了“京密引水渠引水流量不能满足泄洪要求”这一情境，但并未提供具体的处理步骤或替代调度方案，仅作为背景信息存在 [G1]。

证据覆盖范围：文本证据（[T1]-[T6]）提供了密云水库的调度规程、超标准洪水预案及具体水位流量关系；图谱证据（[G1]-[G6]）主要提供了洪水调度示例数据及京密引水渠与调节池的调度关系，但未提供针对引水流量不足的具体处理措施。


## 8. 最小回答验证

验证不是判断“模型写得好不好看”，而是检查几个硬条件：

1. 回答是否为空。
2. 是否引用了证据编号。
3. 引用编号是否真的存在。
4. 如果没有引用，是否明确承认证据不足。

这只是最低门槛。生产系统还会增加：权限校验、敏感信息过滤、事实一致性评估、引用片段回查等。


In [8]:
def extract_citations(answer: str) -> list[str]:
    return re.findall(r'\[([TG]\d+)\]', answer)


def validate_answer(answer: str, citation_map: dict) -> dict:
    citations = extract_citations(answer)
    known = set(citation_map)
    used = set(citations)
    unknown = sorted(used - known)
    says_insufficient = '证据不足' in answer

    checks = {
        'not_empty': bool(answer.strip()),
        'has_citation_or_insufficient': bool(citations) or says_insufficient,
        'all_citations_known': not unknown,
        'citation_count': len(citations),
        'unknown_citations': unknown,
    }
    checks['passed'] = all(
        checks[name]
        for name in ['not_empty', 'has_citation_or_insufficient', 'all_citations_known']
    )
    return checks

validation = validate_answer(answer, citation_map)
pprint(validation)

{'all_citations_known': True,
 'citation_count': 10,
 'has_citation_or_insufficient': True,
 'not_empty': True,
 'passed': True,
 'unknown_citations': []}


## 9. 把引用反查回证据

回答里的 `[T1]`、`[G1]` 不能只停留在文本标记上。

真正的业务系统应该能从引用编号反查到：

- 原始文件
- 页码
- chunk_id
- 图谱关系
- 证据原文

这就是“可追溯回答”的基础。


In [9]:
def resolve_used_citations(answer: str, citation_map: dict) -> list[dict]:
    resolved = []
    for citation_id in sorted(set(extract_citations(answer))):
        item = citation_map.get(citation_id)
        if item is not None:
            resolved.append({'citation_id': citation_id, **item})
    return resolved

used_citations = resolve_used_citations(answer, citation_map)
pprint(used_citations)

[{'chunk_id': '63b7d4d0675426b5_chunk_0030',
  'citation_id': 'G1',
  'object': '京密引水渠',
  'page_end': 25,
  'page_start': 25,
  'predicate': '调度',
  'subject': '调节池',
  'type': 'graph'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0024',
  'citation_id': 'G6',
  'object': '预泄流量200m3/s, 预泄水量0.35 亿m3',
  'page_end': 19,
  'page_start': 19,
  'predicate': '结合',
  'subject': '3 年一遇洪水控泄',
  'type': 'graph'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0029',
  'citation_id': 'T1',
  'page_end': 24,
  'page_start': 24,
  'type': 'text'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0017',
  'citation_id': 'T2',
  'page_end': 12,
  'page_start': 12,
  'type': 'text'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0041',
  'citation_id': 'T6',
  'page_end': 36,
  'page_start': 36,
  'type': 'text'}]


## 10. 保存问答结果

到这里，RAG 问答的最小闭环已经形成：

```text
PDF -> chunk -> embedding/ES -> triples/Neo4j -> hybrid retrieval -> evidence package -> answer -> validation
```


In [10]:
qa_result = {
    'question': evidence_package['question'],
    'answer': answer,
    'answer_source': answer_source,
    'validation': validation,
    'used_citations': used_citations,
    'evidence_package_path': str(evidence_package_path),
}

qa_result_path.write_text(json.dumps(qa_result, ensure_ascii=False, indent=2), encoding='utf-8')

print('qa_result_path:', qa_result_path)
print('size KB:', round(qa_result_path.stat().st_size / 1024, 2))

qa_result_path: /home/dev/bxc/fastapi-study/notebooks/rag/generated/63b7d4d0675426b5/qa_result.json
size KB: 3.24


## 11. 本课小结

这一课你应该记住：

1. RAG 的最后一步不是“把召回结果塞给模型”，而是先把证据压缩成模型能使用的上下文。
2. 回答必须带证据编号，否则无法追溯。
3. 图谱证据适合补充结构关系，但不能替代原文证据。
4. 小模型可以用于问答，但必须配验证器。
5. 最低验证不是质量评测，而是上线前的底线检查。

下一步如果继续扩展，可以把 002-008 串成一个 FastAPI 接口：上传 PDF、入库、检索、问答、返回引用来源。
